## Import Libraries and Data Loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [18]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

In [3]:
df = pd.read_csv('garments_worker_productivity.csv')

## Data Audit

In [7]:
# Inspect column nams and data types (dtypes)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

In [6]:
# Quick summary of numerical columns
df.describe()

,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
count,1197.000000,1197.000000,1197.000000,691.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000
mean,6.426901,0.729632,15.062172,1190.465991,4567.460317,38.210526,0.730159,0.369256,0.150376,34.609858,0.735091
std,3.463963,0.097891,10.943219,1837.455001,3348.823563,160.182643,12.709757,3.268987,0.427848,22.197687,0.174488
min,1.000000,0.070000,2.900000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.233705
25%,3.000000,0.700000,3.940000,774.500000,1440.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.650307
50%,6.000000,0.750000,15.260000,1039.000000,3960.000000,0.000000,0.000000,0.000000,0.000000,34.000000,0.773333
75%,9.000000,0.800000,24.260000,1252.500000,6960.000000,50.000000,0.000000,0.000000,0.000000,57.000000,0.850253
max,12.000000,0.800000,54.560000,23122.000000,25920.000000,3600.000000,300.000000,45.000000,2.000000,89.000000,1.120437


In [ ]:
# Separate numerical and categorical columns
cat_cols = df.select_dtypes(include=['str']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

In [24]:
# Inspect categorical and numerical distribution
print(f"There are {len(cat_cols)} categorical columns, which are:\n", cat_cols)
print(f"There are {len(num_cols)} numerical columns, which are:\n", num_cols)

There are 4 categorical columns, which are:
 ['date', 'quarter', 'department', 'day']
There are 11 numerical columns, which are:
 ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']


In [ ]:
# Check missing values
df.isnull().sum().sort_values(ascending=False)

wip                      506
quarter                    0
date                       0
day                        0
team                       0
targeted_productivity      0
department                 0
smv                        0
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64

In [29]:
# Sample class balance
if 'targeted_productivity' in df.columns:
    print(df['targeted_productivity'].value_counts(normalize=True))

targeted_productivity
0.80    0.451128
0.70    0.202172
0.75    0.180451
0.65    0.052632
0.60    0.047619
0.50    0.040936
0.35    0.022556
0.40    0.001671
0.07    0.000835
Name: proportion, dtype: float64


## Basic Data Cleaning

In [31]:
# Drop duplicate rows
df = df.drop_duplicates()

In [ ]:
# Trim whitespace in categorical columns
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [44]:
# Detect outliers in numerical columns
iqr_mult = 1.5
print("Number of outliers")
for col in num_cols:
    if col in df.columns:
        q1, q3 = df[col].quantile([0.25,0.75])
        iqr = q3 - q1
        lower, upper = q1 - iqr * iqr_mult, q3 + iqr * iqr_mult
        outliers = len(df[(df[col]<lower) | (df[col]>upper)])
        print(f"{col}: {outliers}")

Number of outliers
team: 0
targeted_productivity: 79
smv: 0
wip: 22
over_time: 1
incentive: 11
idle_time: 18
idle_men: 18
no_of_style_change: 147
no_of_workers: 0
actual_productivity: 54
